# 4. Segmentación de Clientes

Este notebook tiene como objetivo cumplir con el **Punto 4** de la práctica, realizando un análisis profundo de los datos para agrupar a los clientes según diferentes criterios y encontrar patrones de compra.

Los análisis a realizar son:
* **4.a:** Agrupar clientes por rangos de edad.
* **4.b:** Comparar el comportamiento de compra entre géneros.
* **4.c:** Analizar el impacto del uso de boletines y vales de descuento.

## 1. Preparación del Entorno y Conexión a la Base de Datos

En esta celda preparamos todo lo necesario para correr el análisis. Si usamos **Google Colab**, el código instalará las librerías necesarias y pedirá las credenciales de la base de datos de la nube. Si lo corremos en **Jupyter local** (nuestra computadora), tomará automáticamente el archivo `.env` que ya tenemos configurado.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

EN_COLAB = 'google.colab' in sys.modules
REPOSITORIO = 'https://github.com/AngelMendoz/SOG2-2S26_grupo8.git'

if EN_COLAB:
    # Instalación de librerías para Colab
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy>=1.26', 'pandas>=2.2', 'psycopg2-binary>=2.9', 'python-dotenv>=1.0', 'SQLAlchemy>=2.0', 'seaborn>=0.13', 'matplotlib>=3.8'], check=True)
    raiz_repositorio = Path('/content/SOG2-2S26_grupo8')
    if not raiz_repositorio.is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', REPOSITORIO, str(raiz_repositorio)], check=True)
    RAIZ_PRACTICA = raiz_repositorio / 'Practica_1'
    
    from google.colab import userdata
    secretos = {nombre: userdata.get(nombre) for nombre in ('DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD')}
    (RAIZ_PRACTICA / '.env').write_text(''.join(f'{k}={v}\n' for k, v in secretos.items()), encoding='utf-8')
else:
    # Configuración de rutas para ejecución local
    RAIZ_PRACTICA = Path.cwd().resolve()
    if not (RAIZ_PRACTICA / 'app').is_dir():
        RAIZ_PRACTICA = RAIZ_PRACTICA.parent

print(f'Entorno listo. Estamos trabajando en: {RAIZ_PRACTICA}')

## 2. Extracción de los Datos (Solo lectura)

Reutilizaremos la conexión a la base de datos de PostgreSQL que se usó en el análisis exploratorio. Obtendremos las tablas completas de `clientes` y `compras`.

In [ ]:
if str(RAIZ_PRACTICA) not in sys.path:
    sys.path.insert(0, str(RAIZ_PRACTICA))

from app.analisis.punto_02 import crear_motor, obtener_datos
from app.analisis.punto_04 import segmentar_por_edad, comportamiento_por_genero, impacto_boletines_vales
from IPython.display import display

# Conectar a la Base de Datos y jalar las tablas
motor = crear_motor(RAIZ_PRACTICA / '.env')
clientes, compras, control = obtener_datos(motor)

# Creamos una carpeta donde se guardarán las fotos de las gráficas
directorio_resultados = RAIZ_PRACTICA / '04-Segmentacion-clientes' / 'resultados'
print("Datos descargados con éxito desde PostgreSQL.")

## 4.a Segmentación por Edad

Agruparemos a los clientes en rangos (Ej. 18-25, 26-35) para analizar cuál es el grupo que más nos compra y cuánto dinero gastan en promedio.

In [ ]:
res_edad = segmentar_por_edad(clientes, directorio_resultados)
display(res_edad)

## 4.b Comportamiento por Género

Verificaremos si existe alguna tendencia de compra distinta entre hombres y mujeres. El `1` se clasifica como Femenino y el `0` como Masculino.

In [ ]:
res_genero = comportamiento_por_genero(clientes, directorio_resultados)
display(res_genero)

## 4.c Impacto de Boletines y Vales

A continuación, agruparemos las compras para analizar qué tanto impacto tiene el hecho de que un cliente reciba el boletín (correo) o haya aplicado un vale de descuento.

In [ ]:
res_boletin_vale = impacto_boletines_vales(compras, directorio_resultados)
display(res_boletin_vale)